In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
mcp = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_prior_med_augmented.csv')
cl = pd.read_csv(f'{DATA_DIR}/class_subtype_lookup.csv')

In [ ]:
cl['antibiotic'] = cl['antibiotic'].str.upper()
cl

In [ ]:
class_exposure = mcp.merge(cl,
                           left_on = ['medication_name'],
                           right_on = ['antibiotic'])
class_exposure = class_exposure[['anon_id','pat_enc_csn_id_coded','order_proc_id_coded',
                                 'order_time_jittered_utc','medication_category','medication_name',
                                 'antibiotic_class','medicationOrderDTS','medication_time_to_cultureTime']]
class_exposure

In [ ]:
class_exposure.to_csv(f'{WORK_DIR}/microbiology_cultures_antibiotic_class_exposure_augmented.csv',
                      index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = class_exposure.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_antibiotic_class_exposure_augmented.csv',
                        index = False)